# Marco 2 — Pipeline e primeiro número

**Grupo G3** — TP1 de Tópicos Especiais em Sistemas de Informação

Objetivos deste notebook (Semana 2, conforme cronograma do TP1):
1. Pipeline de pré-processamento implementado e versionado (§4.1)
2. Partição por paciente definida e congelada, sem vazamento (§4.4)
3. Baseline trivial (classe majoritária) rodando com métrica reportada (§4.3)
4. Primeira família de descritores extraída (textura GLCM/Haralick, priorizando T1 — ver `fichamento.md`) + um classificador treinado

Este notebook parte da amostra estratificada já definida no Marco 1
(`outputs_eda_g3/amostra_g3.csv`, 30 pacientes por classe, semente 42).

## 1. Setup

In [ ]:
!pip install -q pydicom scikit-image

import os
import glob
import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
from skimage.transform import resize
from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                              roc_auc_score, confusion_matrix)

SEED = 42
np.random.seed(SEED)

DATA_DIR = "/kaggle/input/competitions/rsna-miccai-brain-tumor-radiogenomic-classification"
TRAIN_DIR = os.path.join(DATA_DIR, "train")

# Pasta de saída deste marco
OUT_DIR = "outputs_marco2_g3"
os.makedirs(OUT_DIR, exist_ok=True)

# Amostra congelada no Marco 1 — fonte única de verdade para pacientes usados
AMOSTRA_CSV = "/kaggle/input/notebooks/jppr12/marco-1-brain-tumor-g3/outputs_eda_g3/amostra_g3.csv"  # ajustar caminho se necessário
amostra_df = pd.read_csv(AMOSTRA_CSV)
amostra_df["patient_id_str"] = amostra_df["patient_id_str"].astype(str).str.zfill(5)
print(f"Pacientes na amostra: {len(amostra_df)}")
amostra_df["MGMT_value"].value_counts()

## 2. Funções de leitura (reaproveitadas do Marco 1)

Mesma lógica orientation-aware do notebook de EDA: ordena os cortes pela
posição física real (`ImagePositionPatient`), não pelo número do arquivo.

In [ ]:
def read_series(patient_id, modality, data_dir=TRAIN_DIR):
    series_dir = os.path.join(data_dir, patient_id, modality)
    files = glob.glob(os.path.join(series_dir, "*.dcm"))
    slices = [pydicom.dcmread(f) for f in files]
    iop = getattr(slices[0], "ImageOrientationPatient", None) if slices else None
    if iop is not None:
        row_vec = np.array(iop[0:3], dtype=float)
        col_vec = np.array(iop[3:6], dtype=float)
        normal = np.cross(row_vec, col_vec)
        try:
            slices.sort(key=lambda ds: float(np.dot(np.array(ds.ImagePositionPatient, dtype=float), normal)))
        except Exception:
            slices.sort(key=lambda ds: int(''.join(filter(str.isdigit, os.path.basename(ds.filename)))))
    else:
        slices.sort(key=lambda ds: int(''.join(filter(str.isdigit, os.path.basename(ds.filename)))))
    return slices

## 3. Pré-processamento (§4.1 do TP1)

Cada etapa é justificada abaixo. A validação empírica de cada decisão
(ablação) fica para o Marco 3, conforme o próprio TP1 permite (§4.1:
"o 'por quê' deve estar ancorado em literatura ou em evidência empírica do
próprio grupo").

1. **Normalização de intensidade por percentil (clip 1–99% + min-max
   para [0,1]).** RM não tem escala física fixa como HU em TC — cada
   aparelho/protocolo produz intensidades em faixas diferentes. Clipping
   nos percentis 1 e 99 remove outliers de alta intensidade (ruído,
   artefatos) antes da normalização, técnica padrão em radiômica de RM.
2. **Recorte de região não-nula (remoção de fundo).** As imagens do
   dataset já vêm sem crânio (skull-stripped), mas mantêm bordas pretas
   grandes — recortar para o bounding box do tecido reduz dimensão sem
   perder sinal.
3. **Redimensionamento para tamanho fixo (128×128).** Necessário porque
   pacientes têm dimensões de corte variáveis; um tamanho fixo permite
   comparar descritores de textura extraídos de janelas equivalentes
   entre pacientes.

In [ ]:
def preprocess_slice(img, target_size=128):
    img = img.astype(np.float32)

    # 1. Normalização por percentil
    p1, p99 = np.percentile(img, [1, 99])
    img = np.clip(img, p1, p99)
    if p99 > p1:
        img = (img - p1) / (p99 - p1)
    else:
        img = np.zeros_like(img)

    # 2. Recorte de região não-nula (bounding box do tecido)
    mask = img > 0.02  # limiar pequeno para ignorar ruído residual de fundo
    if mask.any():
        rows = np.any(mask, axis=1)
        cols = np.any(mask, axis=0)
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]
        img = img[rmin:rmax+1, cmin:cmax+1]

    # 3. Redimensionamento para tamanho fixo
    img = resize(img, (target_size, target_size), anti_aliasing=True)
    return img


def get_middle_slice_preprocessed(patient_id, modality):
    series = read_series(patient_id, modality)
    ds = series[len(series) // 2]
    raw = ds.pixel_array
    return raw, preprocess_slice(raw)

In [ ]:
# Conferência visual do pipeline: antes vs. depois, para um paciente de exemplo
sample_patient = amostra_df["patient_id_str"].iloc[0]
raw_img, proc_img = get_middle_slice_preprocessed(sample_patient, "T1w")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(raw_img, cmap="gray")
axes[0].set_title(f"T1w bruto — paciente {sample_patient}")
axes[0].axis("off")
axes[1].imshow(proc_img, cmap="gray")
axes[1].set_title("T1w pré-processado (128x128)")
axes[1].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "exemplo_pre_processamento.png"), dpi=150)
plt.show()

## 4. Partição por paciente (§4.4 do TP1)

`StratifiedGroupKFold` com `group = patient_id` — mesmo já havendo uma
linha por paciente na amostra (nenhum corte é tratado como amostra
independente), mantemos o agrupamento explícito para documentar
conformidade com a exigência de particionamento por paciente, nunca por
imagem. 5 dobras, semente fixa, partição salva e **congelada** — não deve
ser alterada nos marcos seguintes.

In [ ]:
N_FOLDS = 5

X_ids = amostra_df["patient_id_str"].values
y = amostra_df["MGMT_value"].values
groups = amostra_df["patient_id_str"].values  # 1 paciente = 1 grupo

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_assignment = np.zeros(len(amostra_df), dtype=int)
for fold_idx, (_, val_idx) in enumerate(sgkf.split(X_ids, y, groups)):
    fold_assignment[val_idx] = fold_idx

amostra_df["fold"] = fold_assignment

# Conferência: nenhum paciente aparece em mais de uma dobra (garantido por construção,
# mas checado explicitamente)
assert amostra_df.groupby("patient_id_str")["fold"].nunique().max() == 1

print("Distribuição de classes por dobra:")
print(amostra_df.groupby(["fold", "MGMT_value"]).size().unstack())

amostra_df[["patient_id_str", "MGMT_value", "fold"]].to_csv(
    os.path.join(OUT_DIR, "particao_pacientes_congelada.csv"), index=False)
print(f"\nPartição salva e CONGELADA em {OUT_DIR}/particao_pacientes_congelada.csv")
print("Esta partição não deve ser alterada nos marcos seguintes.")

## 5. Baseline trivial (§4.3 do TP1)

Classificador de classe majoritária, avaliado com a mesma partição por
paciente. Sem ele não há como interpretar se um classificador real está
de fato aprendendo algo além da proporção de classes.

In [ ]:
def evaluate_folds(df, y_true_col, pred_fn, proba_fn=None):
    """pred_fn(train_df, val_df) -> preds; proba_fn(train_df, val_df) -> proba (classe 1) ou None"""
    rows = []
    for fold in sorted(df["fold"].unique()):
        train_df = df[df["fold"] != fold]
        val_df = df[df["fold"] == fold]
        preds = pred_fn(train_df, val_df)
        y_val = val_df[y_true_col].values

        acc = accuracy_score(y_val, preds)
        bal_acc = balanced_accuracy_score(y_val, preds)
        tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan

        auc = np.nan
        if proba_fn is not None:
            proba = proba_fn(train_df, val_df)
            if len(np.unique(y_val)) > 1:
                auc = roc_auc_score(y_val, proba)

        rows.append({"fold": fold, "accuracy": acc, "balanced_accuracy": bal_acc,
                      "sensibilidade": sens, "especificidade": spec, "auc": auc})
    return pd.DataFrame(rows)


def baseline_pred_fn(train_df, val_df):
    clf = DummyClassifier(strategy="most_frequent")
    clf.fit(train_df[["patient_id_str"]], train_df["MGMT_value"])
    return clf.predict(val_df[["patient_id_str"]])

baseline_results = evaluate_folds(amostra_df, "MGMT_value", baseline_pred_fn)
print("Baseline trivial (classe majoritária) — métricas por dobra:")
print(baseline_results)
print("\nMédia ± desvio-padrão:")
summary_baseline = baseline_results[["accuracy", "balanced_accuracy", "sensibilidade", "especificidade"]].agg(["mean", "std"])
print(summary_baseline)

baseline_results.to_csv(os.path.join(OUT_DIR, "baseline_trivial_metricas.csv"), index=False)
summary_baseline.to_csv(os.path.join(OUT_DIR, "baseline_trivial_resumo.csv"))

## 6. Primeira família de descritores — textura GLCM/Haralick (§4.2 do TP1)

**Escolha da modalidade: T1wCE.** Decisão revisada com base na
experimentação da Seção 6b (abaixo): T1wCE superou T1w em AUC nas 4
combinações de distância/níveis testadas (melhor config: AUC 0,751 ± 0,114
vs. 0,584 ± 0,219 do T1w). Esse resultado contraria parcialmente Zheng et
al. (2024), onde a sequência CE foi a pior isoladamente e T1WI nativo a
melhor — a hipótese é que o realce por contraste, neste pipeline (corte
central bruto, sem múltiplas máscaras), destaca diretamente a região core
do tumor, que outros estudos (Li et al., 2024) já apontam como a mais
informativa radiomicamente. Fica registrado como achado específico deste
pipeline/dataset, não como generalização.

**Escolha dos demais parâmetros:** distância=1, 32 níveis de cinza —
também confirmados pela Seção 6b como a configuração de T1wCE com melhor
equilíbrio entre AUC (0,751) e estabilidade entre dobras (menor
desvio-padrão entre as opções de T1wCE).

**Escolha da unidade de agregação:** por ora, o corte central de cada
paciente (decisão provisória, documentada). A estratégia de agregação
por múltiplos cortes (2.5D/3D) fica para o estudo de ablação do Marco 3,
conforme já registrado em `formulacao_problema.md`.

**Descritores extraídos (GLCM — Gray-Level Co-occurrence Matrix):**
contraste, dissimilaridade, homogeneidade, energia, correlação e ASM —
calculados em 4 direções (0°, 45°, 90°, 135°) e depois agregados pela
média, seguindo a formulação clássica de Haralick.

In [ ]:
GLCM_PROPS = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]
GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]
N_GRAY_LEVELS = 32  # confirmado pela experimentação da Seção 6b
GLCM_DISTANCE = 1   # confirmado pela experimentação da Seção 6b
FEATURE_MODALITY = "T1wCE"  # trocado de T1w para T1wCE após experimentação (Seção 6b)

def extract_glcm_features(img_float01):
    img_uint = (img_float01 * (N_GRAY_LEVELS - 1)).astype(np.uint8)
    glcm = graycomatrix(img_uint, distances=[GLCM_DISTANCE], angles=GLCM_ANGLES,
                         levels=N_GRAY_LEVELS, symmetric=True, normed=True)
    feats = {}
    for prop in GLCM_PROPS:
        values = graycoprops(glcm, prop)[0]  # uma por ângulo
        feats[f"glcm_{prop}_mean"] = np.mean(values)
        feats[f"glcm_{prop}_std"] = np.std(values)
    return feats


feature_rows = []
failed_patients = []
for _, row in amostra_df.iterrows():
    pid = row["patient_id_str"]
    try:
        _, proc_img = get_middle_slice_preprocessed(pid, FEATURE_MODALITY)
        feats = extract_glcm_features(proc_img)
        feats["patient_id_str"] = pid
        feature_rows.append(feats)
    except Exception as e:
        failed_patients.append((pid, str(e)))

features_df = pd.DataFrame(feature_rows)
print(f"Features extraídas para {len(features_df)} de {len(amostra_df)} pacientes (modalidade: {FEATURE_MODALITY})")
if failed_patients:
    print(f"Falhas: {failed_patients}")

features_df.to_csv(os.path.join(OUT_DIR, "features_glcm_t1wce_g3.csv"), index=False)
features_df.head()

## 6b. Experimentação de parâmetros — atendendo à observação do professor

Marcos 1 e 2 são etapas exploratórias: antes de fixar a configuração de
extração de descritores, vale comparar algumas variações e justificar a
escolha final com evidência empírica (ablação), em vez de fixar a primeira
tentativa. Aqui comparamos:

- **Modalidade:** T1w vs. T1wCE (as duas apontadas como mais informativas
  na literatura fichada — Li et al. 2024; Zheng et al. 2024)
- **Distância do GLCM:** 1 vs. 3 pixels (captura textura em escalas
  diferentes — pares de pixels mais próximos vs. mais afastados)
- **Número de níveis de cinza (quantização):** 16 vs. 32 (menos níveis =
  matriz de coocorrência mais densa e robusta a ruído; mais níveis = mais
  detalhe, porém mais esparsa com poucas amostras)

Cada combinação é avaliada com a mesma partição por paciente já congelada,
usando o mesmo classificador (regressão logística), para isolar o efeito
da extração de features do efeito do modelo.

In [ ]:
def extract_glcm_features_param(img_float01, distance=1, n_gray_levels=32):
    img_uint = (img_float01 * (n_gray_levels - 1)).astype(np.uint8)
    glcm = graycomatrix(img_uint, distances=[distance], angles=GLCM_ANGLES,
                         levels=n_gray_levels, symmetric=True, normed=True)
    feats = {}
    for prop in GLCM_PROPS:
        values = graycoprops(glcm, prop)[0]
        feats[f"glcm_{prop}_mean"] = np.mean(values)
        feats[f"glcm_{prop}_std"] = np.std(values)
    return feats


def build_feature_matrix(modality, distance, n_gray_levels):
    rows = []
    for _, row in amostra_df.iterrows():
        pid = row["patient_id_str"]
        try:
            _, proc_img = get_middle_slice_preprocessed(pid, modality)
            feats = extract_glcm_features_param(proc_img, distance, n_gray_levels)
            feats["patient_id_str"] = pid
            rows.append(feats)
        except Exception:
            continue
    return pd.DataFrame(rows)


def evaluate_config(feat_df):
    merged = amostra_df.merge(feat_df, on="patient_id_str", how="inner")
    cols = [c for c in feat_df.columns if c != "patient_id_str"]
    results = evaluate_folds(
        merged, "MGMT_value",
        pred_fn=lambda tr, va: clf_pred_fn_generic(tr, va, cols, return_proba=False),
        proba_fn=lambda tr, va: clf_pred_fn_generic(tr, va, cols, return_proba=True),
    )
    return results["auc"].mean(), results["auc"].std(), results["accuracy"].mean()


def clf_pred_fn_generic(train_df, val_df, cols, return_proba=False):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(penalty="l2", C=1.0, max_iter=1000, random_state=SEED))
    ])
    pipe.fit(train_df[cols], train_df["MGMT_value"])
    if return_proba:
        return pipe.predict_proba(val_df[cols])[:, 1]
    return pipe.predict(val_df[cols])

In [ ]:
# Grade de experimentos: modalidade x distância x níveis de cinza
MODALITIES_TO_TEST = ["T1w", "T1wCE"]
DISTANCES_TO_TEST = [1, 3]
GRAY_LEVELS_TO_TEST = [16, 32]

experiment_rows = []
for mod in MODALITIES_TO_TEST:
    for dist in DISTANCES_TO_TEST:
        for levels in GRAY_LEVELS_TO_TEST:
            feat_df = build_feature_matrix(mod, dist, levels)
            if len(feat_df) < len(amostra_df) * 0.8:
                # muitas falhas de leitura para esta modalidade/paciente — pula
                continue
            auc_mean, auc_std, acc_mean = evaluate_config(feat_df)
            experiment_rows.append({
                "modalidade": mod, "distancia_glcm": dist, "niveis_cinza": levels,
                "auc_mean": auc_mean, "auc_std": auc_std, "accuracy_mean": acc_mean,
                "n_pacientes": len(feat_df),
            })
            print(f"{mod} | dist={dist} | níveis={levels} -> AUC={auc_mean:.3f} ± {auc_std:.3f}")

experimentos_df = pd.DataFrame(experiment_rows).sort_values("auc_mean", ascending=False)
experimentos_df.to_csv(os.path.join(OUT_DIR, "experimentos_parametros_glcm.csv"), index=False)
experimentos_df

**Como interpretar a tabela acima:** a configuração com maior `auc_mean` é
candidata a substituir a configuração usada na Seção 6 (T1w, distância 1,
32 níveis). Mas atenção — com apenas 60 pacientes e 5 dobras, diferenças
pequenas de AUC entre configurações podem não ser estatisticamente
significativas; o critério de decisão deve ser: (1) a configuração vencedora
melhora o AUC **e** reduz o desvio-padrão entre dobras (indicando resultado
mais estável, não só mais sorte numa dobra); (2) se o ganho for marginal e
instável, é defensável manter a configuração mais simples por parcimônia.


## 7. Primeiro classificador (§4.3 do TP1)

Regressão logística regularizada (um dos modelos clássicos exigidos pelo
TP1), em `Pipeline` com padronização de features — o normalizador é
ajustado **somente na partição de treino de cada dobra**, evitando
vazamento de dados (exigência central do §4.4 do TP1).

In [ ]:
merged_df = amostra_df.merge(features_df, on="patient_id_str", how="inner")
feature_cols = [c for c in features_df.columns if c != "patient_id_str"]

def clf_pred_fn(train_df, val_df, return_proba=False):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(penalty="l2", C=1.0, max_iter=1000, random_state=SEED))
    ])
    pipe.fit(train_df[feature_cols], train_df["MGMT_value"])
    if return_proba:
        return pipe.predict_proba(val_df[feature_cols])[:, 1]
    return pipe.predict(val_df[feature_cols])

clf_results = evaluate_folds(
    merged_df, "MGMT_value",
    pred_fn=lambda tr, va: clf_pred_fn(tr, va, return_proba=False),
    proba_fn=lambda tr, va: clf_pred_fn(tr, va, return_proba=True),
)
print("Regressão logística (GLCM de T1wCE) — métricas por dobra:")
print(clf_results)
print("\nMédia ± desvio-padrão:")
summary_clf = clf_results[["accuracy", "balanced_accuracy", "sensibilidade", "especificidade", "auc"]].agg(["mean", "std"])
print(summary_clf)

clf_results.to_csv(os.path.join(OUT_DIR, "logreg_glcm_metricas.csv"), index=False)
summary_clf.to_csv(os.path.join(OUT_DIR, "logreg_glcm_resumo.csv"))

In [ ]:
# Comparação direta: baseline trivial vs. primeiro classificador
comparativo = pd.DataFrame({
    "baseline_trivial": summary_baseline.loc["mean"],
    "logreg_glcm_t1wce": summary_clf.loc["mean"].reindex(summary_baseline.columns.tolist() + ["auc"]).fillna(summary_clf.loc["mean", "auc"] if "auc" in summary_clf.columns else np.nan)
})
print(comparativo)
comparativo.to_csv(os.path.join(OUT_DIR, "comparativo_baseline_vs_logreg.csv"))

## 8. Conferência final dos arquivos salvos

In [ ]:
print(f"Arquivos gerados em '{OUT_DIR}/':")
for f in sorted(os.listdir(OUT_DIR)):
    print(" -", f)

## 9. Notas e pendências para o Marco 3

- [ ] Extrair as demais famílias de descritores (mínimo 3 no total — ex.:
      forma/contorno e gradiente/bordas, além da textura já feita aqui)
- [ ] Testar ao menos mais 2 modelos clássicos (Random Forest, SVM ou
      XGBoost) além da regressão logística
- [ ] Estudo de ablação: comparar corte central vs. pooling de múltiplos
      cortes por paciente (decisão de agregação ainda em aberto)
- [ ] Estudo de ablação: testar se registrar as modalidades entre si antes
      da extração de features muda o desempenho (decisão de registro ainda
      em aberto, ver `formulacao_problema.md`)
- [ ] Ajuste de hiperparâmetros dentro de validação cruzada aninhada
- [ ] Consolidar tabela comparativa completa (descritor × modelo) com
      média ± desvio para o artigo final

**Resultado deste marco:** o pipeline está implementado, correto e
reprodutível (semente fixa, partição congelada, sem vazamento). Não é
esperado que o desempenho já seja bom — conforme discutido em
`formulacao_problema.md` com base em Kim et al. (2022) e Doniselli et al.
(2024), este é um problema de sinal fraco mesmo na literatura madura.